# Load action annotations and align with events
Here is the example script to load the action segments and align the action segments with event slices by timestamps. Firstly load the needed libraries.

In [1]:
import h5py
import hdf5plugin
import pandas as pd
import numpy as np
from event_reader.eventslicer import EventSlicer

Then input the path to the action annotation file, we use pandas to load the csv file.

In [ ]:
# load action labels
action_label_file = "" # path to the Annotations/actions/cooking_activity_action.csv
action_labels = pd.read_csv(action_label_file)

Then input the path of event file which corrsponds to the action annotation. We suggest you to add code to check whether the annotation file and event file are matched (same activity under same recording session).

In [ ]:
# load events
event_file = "" # path to the dataset/LeftEvent/LeftEvent.hdf5 or dataset/RightEvent/RightEvent.hdf5
event_loader = EventSlicer(h5py.File(event_file, 'r'))

####
## code to check the whether the annotation and data are matched
####

The following code can automatically load the aligned event slice and action label, which is saved in ***aligned_data***. We also need to make sure the defected time periods are not loaded as mentioned in [`read_event_file.ipynb`](read_event_file.ipynb). After that, you can start your own data processing code.

In [ ]:
# load defected time period
defect_stats = pd.read_csv('defect_data_stats.csv')
def check_data(event_file, start_time, end_time):
    for i, s in defect_stats["Session_ID"].items():
        a = defect_stats.loc[i, "Activity"]
        if s in event_file and a in event_file:
            # the load event file contains defected parts
            # next is to check whether the input timestamps are in the defect period
            
            # full:   |                                 |
            # defect:               |      |
            # input:       |   |       or        |   |
            if end_time <= defect_stats.loc[i, "start_t"] or start_time >= defect_stats.loc[i, "end_t"]:
                checked_event = [[start_time, end_time]]
                
            # full:   |                                 |
            # defect:               |      |
            # input:            |      |        
            elif end_time <= defect_stats.loc[i, "end_t"] and start_time < defect_stats.loc[i, "start_t"]:
                end_time = defect_stats.loc[i, "start_t"]
                checked_event = [[start_time, end_time]]
                
            # full:   |                                 |
            # defect:               |      |
            # input:                    |      | 
            elif end_time > defect_stats.loc[i, "end_t"] and start_time >= defect_stats.loc[i, "start_t"]:
                start_time = defect_stats.loc[i, "end_t"]
                checked_event = [[start_time, end_time]]
                
            # full:   |                                 |
            # defect:               |      |
            # input:                 |    | 
            elif end_time < defect_stats.loc[i, "end_t"] and start_time > defect_stats.loc[i, "start_t"]:
                checked_event = None

            # full:   |                                 |
            # defect:               |      |
            # input:             |            | 
            else:
               checked_event = [[start_time, defect_stats.loc[i, "start_t"]], [end_time, defect_stats.loc[i, "end_t"]]]
                                 
    return checked_event

# align events and action
for index, label in action_labels["Action_label"].items():
    start_time = action_labels.loc[index, "Global_start_time"]
    end_time = action_labels.loc[index, "Global_end_time"]
    checked_event = check_data(event_file, start_time, end_time)
    # skip defected event
    if checked_event is None:
        continue
    else:
        for [start_ts, end_ts] in checked_event:
            # convert second to microsecond
            start_ts = start_ts * 1e6
            end_ts = start_ts * 1e6
            event_slice = event_loader.get_events(start_time, end_time)
            aligned_data = [label, event_slice] # the alinged action label and event sclice

            #####
            ## your own data processing
            #####